# 01 — Dataset Analysis

Explore the raw KOA-PD-NM dataset before any processing: class balance,
severity breakdown, subject counts, video duration/frame-rate stats, and
a visual sample of raw frames per class.

Run this **before** `02_generate_MHI.ipynb` so you know what you're working
with (in particular: how imbalanced PD is relative to KOA and Healthy).

In [ ]:
import os
import sys

# Make the project's src/ package importable from within notebooks/
sys.path.insert(0, os.path.abspath(".."))

import cv2
import matplotlib.pyplot as plt
import pandas as pd

from src.utils.config import RAW_VIDEOS_DIR, RAW_CLASS_DIRS
from src.utils.helpers import list_videos, extract_subject_id, parse_class_and_severity
from src.preprocessing.video_loader import count_frames

%matplotlib inline

## Build a raw metadata table (no MHI generation yet)

In [ ]:
rows = []
for class_name, folder_name in RAW_CLASS_DIRS.items():
    class_dir = os.path.join(RAW_VIDEOS_DIR, folder_name)
    if not os.path.isdir(class_dir):
        print(f"Missing directory (expected before dataset is placed): {class_dir}")
        continue
    for video_path in list_videos(class_dir):
        cls, severity = parse_class_and_severity(video_path)
        subject_id = extract_subject_id(video_path)
        rows.append({
            "video_path": video_path,
            "class": cls,
            "severity": severity,
            "subject_id": subject_id,
        })

df = pd.DataFrame(rows)
print(f"Found {len(df)} videos.")
df.head()

> If `df` is empty, you haven't placed the KOA-PD-NM videos under
> `data/raw_videos/{KOA,PD,NM}/...` yet — do that first (see the top-level
> README), then re-run this cell.

## Class balance

In [ ]:
class_counts = df["class"].value_counts()
print(class_counts)

fig, ax = plt.subplots(figsize=(5, 4))
class_counts.plot(kind="bar", ax=ax, color=["#4C72B0", "#DD8452", "#55A868"])
ax.set_title("Videos per class")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

Note the class imbalance: PD has far fewer videos than KOA or Healthy in
the base dataset (only 31 PD videos total across all severities vs. 100 KOA
and 60 Healthy). Keep this in mind for stratified splitting and for
interpreting per-class recall later — a model can look good on overall
accuracy while doing poorly on PD specifically.

## Severity breakdown (KOA / PD only)

In [ ]:
severity_counts = (
    df[df["class"].isin(["KOA", "PD"])]
    .groupby(["class", "severity"])
    .size()
    .reset_index(name="count")
)
severity_counts

In [ ]:
pivot = severity_counts.pivot(index="severity", columns="class", values="count").fillna(0)
order = [s for s in ["Early", "Moderate", "Severe"] if s in pivot.index]
pivot = pivot.loc[order]

fig, ax = plt.subplots(figsize=(6, 4))
pivot.plot(kind="bar", ax=ax)
ax.set_title("Severity breakdown by class")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

## Subject counts (this is what actually gets split, not videos)

In [ ]:
subjects_per_class = df.groupby("subject_id")["class"].agg(lambda s: s.value_counts().idxmax())
print("Unique subjects:", subjects_per_class.shape[0])
print(subjects_per_class.value_counts())

videos_per_subject = df.groupby("subject_id").size()
print("\nVideos per subject (should mostly be 2 -- left-to-right and right-to-left walks):")
print(videos_per_subject.value_counts().sort_index())

Confirm most subjects contribute exactly 2 videos. If a subject shows up
with only 1, or with a surprisingly high count, it's worth double-checking
`extract_subject_id()` in `src/utils/helpers.py` handled that filename
correctly -- a wrong subject_id silently breaks the leakage-free split
downstream.

## Video duration / frame-count sanity check

In [ ]:
def video_stats(path):
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    duration = n_frames / fps if fps else float("nan")
    return pd.Series({"fps": fps, "n_frames": n_frames, "width": width, "height": height, "duration_sec": duration})

if len(df) > 0:
    stats_df = df["video_path"].apply(video_stats)
    stats_df = pd.concat([df[["class", "severity", "subject_id"]], stats_df], axis=1)
    display(stats_df.describe())
    stats_df.groupby("class")["duration_sec"].describe()

## Visual sample: one raw frame per class

In [ ]:
fig, axes = plt.subplots(1, len(RAW_CLASS_DIRS), figsize=(4 * len(RAW_CLASS_DIRS), 4))

for ax, class_name in zip(axes, RAW_CLASS_DIRS.keys()):
    subset = df[df["class"] == class_name]
    if subset.empty:
        ax.set_title(f"{class_name} (no videos found)")
        ax.axis("off")
        continue
    sample_path = subset.iloc[0]["video_path"]
    cap = cv2.VideoCapture(sample_path)
    # Grab a frame partway through the walk, not just frame 0 (often a static start pose).
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.set(cv2.CAP_PROP_POS_FRAMES, total // 2)
    ret, frame = cap.read()
    cap.release()
    if ret:
        ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    ax.set_title(class_name)
    ax.axis("off")

plt.tight_layout()
plt.show()

## Next step

Once the class balance, subject counts, and frame stats look sane, move on
to **`02_generate_MHI.ipynb`** to turn these raw videos into Motion History
Images and build the leakage-free train/validation/test split.